In [1]:
import numpy as np
import pandas as pd

SEED = 42
SECONDS_PER_DAY = 24 * 60 * 60

def arrival_rate_per_second(hour, direction):
    morning = np.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
    evening = np.exp(-0.5 * ((hour - 17) / 2) ** 2)
    if direction == "NS":
        per_5min = 8 + 35 * morning + 30 * evening
    else:
        per_5min = 6 + 25 * morning + 40 * evening
    return per_5min / 300   # vehicles per second

def generate_arrivals(seed, num_days=1):
    rng = np.random.default_rng(seed)
    t = np.arange(num_days * SECONDS_PER_DAY)
    hour = (t / 3600) % 24
    ns = rng.random(len(t)) < arrival_rate_per_second(hour, "NS")
    ew = rng.random(len(t)) < arrival_rate_per_second(hour, "EW")
    return ns.astype(int), ew.astype(int)

ns_arrivals, ew_arrivals = generate_arrivals(SEED)

print("Seconds simulated:", len(ns_arrivals))
print("Total NS vehicles:", ns_arrivals.sum())
print("Total EW vehicles:", ew_arrivals.sum())

# Sanity check: busy hour vs quiet hour
print("NS vehicles during 8 AM hour:", ns_arrivals[8*3600:9*3600].sum())
print("NS vehicles during 2 AM hour:", ns_arrivals[2*3600:3*3600].sum())

Seconds simulated: 86400
Total NS vehicles: 5636
Total EW vehicles: 5259
NS vehicles during 8 AM hour: 483
NS vehicles during 2 AM hour: 109


In [2]:
GREEN_FIXED = 30
YELLOW = 3
MIN_GREEN = 10
MAX_GREEN = 60
DISCHARGE_PER_SEC = 0.5   # one vehicle leaves every 2 seconds on green

def fixed_time_controller(state):
    return state["phase_timer"] >= GREEN_FIXED

def rule_based_controller(state):
    queue = state["ns_queue"] if state["phase"] == "NS_GREEN" else state["ew_queue"]
    t = state["phase_timer"]
    return t >= MAX_GREEN or (t >= MIN_GREEN and queue == 0)

def simulate(controller, ns_arrivals, ew_arrivals):
    n = len(ns_arrivals)
    ns_q, ew_q = 0, 0
    phase, phase_timer = "NS_GREEN", 0
    credit, served = 0.0, 0
    ns_log = np.zeros(n, dtype=int)
    ew_log = np.zeros(n, dtype=int)

    for t in range(n):
        # 1. arrivals
        ns_q += ns_arrivals[t]
        ew_q += ew_arrivals[t]

        # 2. discharge on green
        if phase in ("NS_GREEN", "EW_GREEN"):
            credit += DISCHARGE_PER_SEC
            if credit >= 1:
                credit -= 1
                if phase == "NS_GREEN" and ns_q > 0:
                    ns_q -= 1
                    served += 1
                elif phase == "EW_GREEN" and ew_q > 0:
                    ew_q -= 1
                    served += 1

        phase_timer += 1

        # 3. phase switching
        state = {"phase": phase, "phase_timer": phase_timer,
                 "ns_queue": ns_q, "ew_queue": ew_q, "t": t}
        if phase == "NS_GREEN" and controller(state):
            phase, phase_timer = "NS_YELLOW", 0
        elif phase == "NS_YELLOW" and phase_timer >= YELLOW:
            phase, phase_timer = "EW_GREEN", 0
        elif phase == "EW_GREEN" and controller(state):
            phase, phase_timer = "EW_YELLOW", 0
        elif phase == "EW_YELLOW" and phase_timer >= YELLOW:
            phase, phase_timer = "NS_GREEN", 0

        # 4. log
        ns_log[t], ew_log[t] = ns_q, ew_q

    # metrics
    hour = (np.arange(n) / 3600) % 24
    peak = (hour >= 8) & (hour < 9)
    total_arrivals = ns_arrivals.sum() + ew_arrivals.sum()
    total_delay = ns_log.sum() + ew_log.sum()   # vehicle-seconds spent waiting

    return {
        "avg_wait_sec_per_vehicle": round(total_delay / total_arrivals, 2),
        "avg_ns_queue_full_day": round(ns_log.mean(), 2),
        "avg_ew_queue_full_day": round(ew_log.mean(), 2),
        "avg_ns_queue_8am": round(ns_log[peak].mean(), 2),
        "avg_ew_queue_8am": round(ew_log[peak].mean(), 2),
        "max_ns_queue": int(ns_log.max()),
        "max_ew_queue": int(ew_log.max()),
        "vehicles_arrived": int(total_arrivals),
        "vehicles_served": int(served),
    }

print("Simulation engine ready.")

Simulation engine ready.


In [3]:
results = {
    "fixed_time": simulate(fixed_time_controller, ns_arrivals, ew_arrivals),
    "rule_based": simulate(rule_based_controller, ns_arrivals, ew_arrivals),
}

comparison = pd.DataFrame(results).T
comparison

,avg_wait_sec_per_vehicle,avg_ns_queue_full_day,avg_ew_queue_full_day,avg_ns_queue_8am,avg_ew_queue_8am,max_ns_queue,max_ew_queue,vehicles_arrived,vehicles_served
fixed_time,13.26,0.86,0.82,1.99,1.45,12.0,11.0,10895.0,10895.0
rule_based,7.50,0.49,0.45,1.18,0.92,7.0,7.0,10895.0,10895.0


In [ ]:
import os
os.makedirs("../results", exist_ok=True)
comparison.to_csv("../results/controller_comparison_v2.csv")
print("Saved.")